In [1]:
# Load required packages
import numpy as np
from evolution_engine import EvolutionEngine
from memory_system import MemoryType

print(f"NumPy version: {np.__version__}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/xavierhillroy/anaconda3/envs/LGP_VISION/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/xavierhillroy/anaconda3/envs/LGP_VISION/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/xavierhillroy/anaconda3/envs/LGP_VISION/lib/python3.11/site-packages/ipykernel/kernel

NumPy version: 2.3.5


In [2]:
# Load the checkpoint
print("Loading checkpoint...")

checkpoint = EvolutionEngine.load_checkpoint("checkpoints/best_population.pkl")
population = checkpoint['population']
generation = checkpoint['generation']
fitness = checkpoint['fitness']

print(f"✓ Checkpoint loaded successfully!")
print(f"  Generation: {generation}")
print(f"  Best fitness: {fitness:.4f}")
print(f"  Population size: {len(population)}")
print()


Loading checkpoint...
✓ Checkpoint loaded successfully!
  Generation: 42
  Best fitness: 1.0972
  Population size: 500



In [3]:
# Get the best individual from the population
best_individual = population.get_best()

print("="*70)
print("BEST INDIVIDUAL IN POPULATION")
print("="*70)
print(f"ID: {best_individual.id}")
print(f"Fitness: {best_individual.fitness:.4f}")
print(f"Age: {best_individual.age}")
print(f"Program length (total): {len(best_individual.program)}")
print(f"Parent IDs: {best_individual.parent_ids if best_individual.parent_ids else 'None'}")
print()


BEST INDIVIDUAL IN POPULATION
ID: 18041
Fitness: 1.0972
Age: 0
Program length (total): 12
Parent IDs: (15131, 15964)



In [5]:
# Get output registers - typically FlappyBird uses scalar register 7 for action output
# We can check the checkpoint config if available, or use the common default
output_register = 0  # Common default for FlappyBird
output_registers = [(MemoryType.SCALAR, output_register)]

# Calculate effective length and intron ratio
effective_length = best_individual.get_effective_length(output_registers)
intron_ratio = best_individual.get_intron_ratio(output_registers)

print(f"Output registers: {output_registers}")
print(f"Effective length: {effective_length}")
print(f"Intron ratio: {intron_ratio:.3f} ({intron_ratio*100:.1f}% introns)")
print()


Output registers: [(<MemoryType.SCALAR: 'scalar'>, 0)]
Effective length: 1
Intron ratio: 0.917 (91.7% introns)



In [6]:
# Display the full program
print("="*70)
print("PROGRAM INSTRUCTIONS")
print("="*70)
print()

for i, instr in enumerate(best_individual.program.instructions):
    print(f"{i:4d}: {instr}")

print()
print("="*70)
print(f"Total: {len(best_individual.program)} instructions")
print("="*70)


PROGRAM INSTRUCTIONS

   0: matrix[5] = automl_vector_outer(vector[4], vector[2])
   1: scalar[1] = automl_scalar_cos(scalar[3])
   2: matrix[3] = cv_avg_pool(matrix[0], scalar[3])
   3: matrix[2] = cv_dilate(matrix[6], scalar[5])
   4: matrix[2] = automl_matrix_div(matrix[4], matrix[3])
   5: vector[2] = automl_vector_heaviside(vector[2])
   6: scalar[6] = automl_scalar_max(scalar[6], scalar[3])
   7: scalar[5] = automl_scalar_div(scalar[6], scalar[4])
   8: scalar[0] = automl_scalar_tan(scalar[0])
   9: scalar[2] = automl_vector_dot(vector[0], vector[6])
  10: matrix[0] = cv_avg_pool(matrix[1], scalar[0])
  11: vector[6] = automl_matrix_norm_axis1(obs_matrix[-1])

Total: 12 instructions


In [7]:
# Show effective program (with introns removed)
effective_program = best_individual.get_effective_program(output_registers)

print("="*70)
print("EFFECTIVE PROGRAM (INTRONS REMOVED)")
print("="*70)
print()

for i, instr in enumerate(effective_program.instructions):
    print(f"{i:4d}: {instr}")

print()
print("="*70)
print(f"Effective: {len(effective_program)} instructions (out of {len(best_individual.program)} total)")
print("="*70)


EFFECTIVE PROGRAM (INTRONS REMOVED)

   0: scalar[0] = automl_scalar_tan(scalar[0])

Effective: 1 instructions (out of 12 total)


In [8]:
# Display population statistics for context
print("="*70)
print("POPULATION STATISTICS")
print("="*70)
min_fit, mean_fit, max_fit, std_fit = population.compute_statistics()
print(f"Fitness - Min: {min_fit:.4f}, Mean: {mean_fit:.4f}, Max: {max_fit:.4f}, Std: {std_fit:.4f}")

diversity = population.get_diversity_metrics()
print(f"Program length - Mean: {diversity['mean_length']:.1f}, Std: {diversity['std_length']:.1f}")

if population.best_ever is not None:
    print(f"Best ever fitness: {population.best_ever.fitness:.4f} at generation {population.best_ever_generation}")
print("="*70)


POPULATION STATISTICS
Fitness - Min: 0.0200, Mean: 0.2085, Max: 1.0972, Std: 0.1999
Program length - Mean: 10.8, Std: 5.2
Best ever fitness: 1.0972 at generation 42


In [12]:
# Show constants if the individual has any
constants = best_individual.memory
print(constants)



MemoryBank(obs=[0s, 0v, 1m], registers=[8s, 8v, 8m])


In [ ]:
viz_evaluator = FlappyBirdEvaluator(
    env_id="FlappyBird-v0",
    episodes=1,              # just watch one run
    max_steps=500,
    output_register=evolution_evaluator.output_register,
    render_mode="human",     # opens the pygame window
    rng=np.random.default_rng(0),
    patch_strategy=evolution_evaluator.patch_strategy,
    color_channel=evolution_evaluator.color_channel,
    normalize=evolution_evaluator.normalize,
    quantization_factor=evolution_evaluator.quantization_factor,
)

fitness = viz_evaluator.evaluate(best_agent)
print(f"Best agent fitness during visualization: {fitness:.3f}")